# core

> Ergonomic wrapper for pandas_gbq that simplifies loading BigQuery data into DataFrames

This module provides an ergonomic wrapper around `pandas_gbq` to simplify working with BigQuery in pandas. The main functions are:

- `read()` - Load data from BigQuery queries or tables into DataFrames with automatic type conversion
- `to()` - Write DataFrames to BigQuery tables
- `ex()` - Execute queries without returning results (useful for DDL/DML)
- `exists()` - Check if a table exists
- `ensure()` - Read a table if it exists, otherwise create it from a query first
- `pipeline()` - Create multiple tables from a dict with skip/resume support

All functions include verbose timing and size reporting by default.

In [ ]:
#| default_exp core

**Imports**

In [ ]:
from nbdev.showdoc import *

In [ ]:
#| export
from pandas_gbq import read_gbq as _original_read_gbq, to_gbq as _original_to_gbq, Context, context
import pandas as pd, re, time
from decimal import Decimal
from google.cloud import bigquery
from dataclasses import dataclass
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
from google.oauth2 import service_account
from google.auth import default
import os
import json

**Credentials helper** - Get credentials from environment variable for CI/CD, or fall back to ADC

In [ ]:
#| export
def _get_credentials():
    creds_json = os.getenv('GOOGLE_CREDENTIALS_JSON')
    if creds_json: return service_account.Credentials.from_service_account_info(json.loads(creds_json))
    creds_file = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
    if creds_file: return service_account.Credentials.from_service_account_file(creds_file)
    return None


In [ ]:

creds = _get_credentials()
print(f"Credentials: {'from env' if creds else 'using ADC'}")


Credentials: from env


In [ ]:

print('GOOGLE_CREDENTIALS_JSON' in os.environ)
print('GOOGLE_APPLICATION_CREDENTIALS' in os.environ)
if 'GOOGLE_APPLICATION_CREDENTIALS' in os.environ: print(os.environ['GOOGLE_APPLICATION_CREDENTIALS'][:50])


True
False


In [ ]:
creds_json = os.getenv('GOOGLE_CREDENTIALS_JSON')
if creds_json:
    parsed = json.loads(creds_json)
    print('Keys found:', list(parsed.keys()))
    print('Has client_email:', 'client_email' in parsed)
    print('Has token_uri:', 'token_uri' in parsed)

Keys found: ['type', 'project_id', 'private_key_id', 'private_key', 'client_email', 'client_id', 'auth_uri', 'token_uri', 'auth_provider_x509_cert_url', 'client_x509_cert_url', 'universe_domain']
Has client_email: True
Has token_uri: True


In [ ]:
try:
    creds = _get_credentials()
    print(f"Credentials: {creds}")
    if creds is None:
        print("No credentials found - will try Application Default Credentials")
        creds, project = default()
        print(f"ADC found: {creds}")
except Exception as e:
    print(f"Error: {e}")

Credentials: <google.oauth2.service_account.Credentials object>


**Core read function** - Wraps `pandas_gbq.read_gbq` with automatic type conversion and timing

In [ ]:
#| export
def read(
    query_or_table:str, # SQL query string or table reference
    verbose:bool=True, # Print timing and DataFrame info
    convert_dtypes:bool=True, # Apply type conversion to columns
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=["scv_id"], # List of columns to keep as string/object type
    use_bqstorage_api:bool=True, # Use BigQuery Storage API for faster reads
    credentials=None, # Optional explicit credentials
    **kwargs
    ):
    start = time.time()
    creds = credentials or _get_credentials()
    df = _original_read_gbq(query_or_table, use_bqstorage_api=use_bqstorage_api, credentials=creds, **kwargs)
    if convert_dtypes: df = convert_bq_dtypes(df, date_cols=date_cols, str_cols=str_cols)
    read_type = 'table' if re.match(r"""^[`\w\-]+\.[\w\-]+.[\w\-\`]+$""", query_or_table) else 'query'
    if verbose:
        elapsed = time.time() - start
        size_gb = _get_size_gb(df)
        print(f"Loaded {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) from {read_type} in {elapsed:.2f}s")
        print(df.info())
    return df

**Type conversion helper** - Converts BigQuery types to appropriate pandas nullable types

In [ ]:
#| export
def convert_bq_dtypes(
    df:pd.DataFrame, # DataFrame to convert
    date_cols:list=None, # List of columns to convert to datetime
    str_cols:list=None # List of columns to keep as string/object type
):
    "Convert BigQuery data types to pandas-compatible types"
    df = df.copy()
    date_cols = set(date_cols or [])
    str_cols = set(str_cols or [])
    for col in df.columns:
        if col in str_cols: df[col] = df[col].astype('object')
        elif re.search(r"(date|timestamp)", col.lower()) or col in date_cols: df[col] = pd.to_datetime(df[col], errors='coerce')
        elif df[col].dtype == 'float64': df[col] = df[col].astype('Float64')
        elif df[col].dtype == 'int64': df[col] = df[col].astype('Int64')
        elif df[col].dtype == 'object':
            first_val = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
            if first_val is not None and isinstance(first_val, Decimal): df[col] = df[col].astype('Float64')
    return df

**Write function** - Wraps `pandas_gbq.to_gbq` with timing info

In [ ]:
#| export
def to(
    df:pd.DataFrame, # DataFrame to write to BigQuery
    destination_table:str, # Destination table in format 'project.dataset.table' or 'dataset.table'
    verbose:bool=True, # Print timing and data size info
    credentials=None, # Optional explicit credentials
    **kwargs
):
    "Write DataFrame to BigQuery table"
    start = time.time()
    creds = credentials or _get_credentials()
    result = _original_to_gbq(df, destination_table, credentials=creds, **kwargs)
    if verbose:
        elapsed = time.time() - start
        size_gb = _get_size_gb(df)
        print(f"Sent {len(df)} rows × {len(df.columns)} cols ({size_gb:.4f} GB) to {destination_table} in {elapsed:.2f}s")
    return result

**Memory helper** - Calculate DataFrame size in GB

In [ ]:
#| export

def _get_size_gb(df):
    return df.memory_usage(deep=True).sum() / 1024**3


**Execute function** - Run queries without returning results (DDL/DML operations)

In [ ]:
#| export
def ex(
    query:str, # SQL query string
    project_id:str, # Project ID
    verbose:bool=True, # Print timing and processing info
    credentials=None, # Optional explicit credentials
    **kwargs
    ):
    "Execute query in BigQuery without returning results"
    creds = credentials or _get_credentials()
    client = bigquery.Client(project=project_id, credentials=creds, **kwargs)
    start = time.time()
    job = client.query(query)
    result = job.result()
    if verbose:
        elapsed = time.time() - start
        gb_processed = (job.total_bytes_processed or 0) / 1024**3
        rows_affected = job.num_dml_affected_rows if job.num_dml_affected_rows else 0
        cached = " (cached)" if job.cache_hit else ""
        print(f"Processed {gb_processed:.4f} GB{cached}, {rows_affected} rows affected in {elapsed:.2f}s")
    return result

**Table helper class** - Parse and represent BigQuery table references

In [ ]:
#| export
@dataclass
class Table:
    project: str
    dataset: str
    name: str
    
    @classmethod
    def from_id(cls, table_id:str):
        parts = table_id.replace('`','').split('.')
        if len(parts) == 3: return cls(*parts)
        if len(parts) == 2: return cls(os.getenv('GCP_PROJECT'), *parts)
        raise ValueError(f"Invalid table_id: {table_id}")
    
    @property
    def id(self): return f"{self.project}.{self.dataset}.{self.name}"

    def __str__(self): return self.id



**Existence checker** - Check if a BigQuery table exists

In [ ]:
#| export
def exists(
    table_id:str, # Table reference (project.dataset.table or dataset.table)
    project_id:str, # Default project if not in table_id
    credentials=None, # Optional explicit credentials
    verbose:bool=False, # Print existence status
    **kwargs
):
    "Check if a BigQuery table exists"
    if isinstance(table_id, Table): table_id = table_id.id
    elif table_id.count('.') == 1: table_id = f"{project_id}.{table_id}"
    creds = credentials or _get_credentials()
    client = bigquery.Client(credentials=creds, project=project_id, **kwargs)
    try:
        client.get_table(table_id)
        if verbose: print(f"Table {table_id} exists")
        return True
    except NotFound:
        if verbose: print(f"Table {table_id} not found")
        return False

**Ensure function** - Read table if exists, otherwise create from query then read

In [ ]:
#| export
def ensure(
    table_id:str,
    query:str,
    project_id:str,
    force:bool=False,
    verbose:bool=True,
    credentials=None,
    **kwargs
):
    "Read table if exists, otherwise create from query then read"
    creds = credentials or _get_credentials()
    if not exists(table_id, project_id=project_id, credentials=creds, verbose=verbose) or force:
        create_query = f"CREATE OR REPLACE TABLE {table_id} AS {query}"
        ex(create_query, project_id=project_id, credentials=creds, verbose=verbose)
    return read(table_id, verbose=verbose, credentials=creds, **kwargs)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
